In [1]:
# =================================================================================================
# NHANES 2003-2004 Dietary Merge
#
# Purpose:
#   Merge DR1TOT_C and DR2TOT_C into one participant-level Dietary CSV.
#
# Final structure:
#
#   SEQN
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#   [Day 1 variables]
#   [Day 2 variables]
#
# Fidelity rules:
#   - One SEQN per participant
#   - Preserve Dietary decimal text AS IS
#   - Do not round values
#   - Do not convert Dietary measurements to integers
#   - Validate shared variables before removing duplicate copies
#   - Prevent many-to-many merge errors
#
# Files:
#   DR1TOT_C_corrected.csv
#   DR2TOT_C_corrected.csv
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
from pathlib import Path


# =================================================================================================
# 2. FILE PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent

DAY1_FILE = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Dietary"
    / "DR1TOT_C_corrected.csv"
)

DAY2_FILE = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Dietary"
    / "DR2TOT_C_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2003_2004"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_dietary_2003_2004_raw_corrected.csv"
)


# =================================================================================================
# 3. READ CORRECTED CSV FILES AS TEXT
# =================================================================================================

#
# IMPORTANT:
#
# dtype=str prevents pandas from reinterpreting Dietary decimal values.
#
# Therefore values already corrected in:
#
#     DR1TOT_C_corrected.csv
#     DR2TOT_C_corrected.csv
#
# remain textually AS IS during this merge.
#
# Example:
#
#     0.028999999999
#
# remains:
#
#     0.028999999999
#
# rather than being reformatted.
#

day1 = pd.read_csv(
    DAY1_FILE,
    dtype=str,
    keep_default_na=False
)


day2 = pd.read_csv(
    DAY2_FILE,
    dtype=str,
    keep_default_na=False
)


print("=" * 100)
print("NHANES 2003-2004 DIETARY DAY 1 + DAY 2 MERGE")
print("=" * 100)

print(
    "Day 1 shape:",
    day1.shape
)

print(
    "Day 2 shape:",
    day2.shape
)


# =================================================================================================
# 4. STANDARDIZE COLUMN NAMES
# =================================================================================================

day1.columns = (
    day1.columns
    .str.strip()
    .str.upper()
)

day2.columns = (
    day2.columns
    .str.strip()
    .str.upper()
)


# =================================================================================================
# 5. VERIFY SEQN EXISTS
# =================================================================================================

for name, df in [
    ("DR1TOT_C", day1),
    ("DR2TOT_C", day2)
]:

    if "SEQN" not in df.columns:

        raise ValueError(
            f"{name} does not contain SEQN."
        )


print(
    "PASS: SEQN exists in both files."
)


# =================================================================================================
# 6. VERIFY SEQN IS NOT STORED WITH .0
# =================================================================================================

for name, df in [
    ("DR1TOT_C", day1),
    ("DR2TOT_C", day2)
]:

    dot_zero_count = int(
        (
            df["SEQN"].ne("")
            &
            df["SEQN"].str.endswith(".0")
        )
        .sum()
    )


    if dot_zero_count != 0:

        raise ValueError(
            f"{name}: "
            f"{dot_zero_count:,} SEQN values still end in '.0'."
        )


print(
    "PASS: SEQN contains no '.0' in either file."
)


# =================================================================================================
# 7. VERIFY SEQN IS UNIQUE WITHIN EACH FILE
# =================================================================================================

for name, df in [
    ("DR1TOT_C", day1),
    ("DR2TOT_C", day2)
]:

    blank_seqn = int(
        df["SEQN"]
        .eq("")
        .sum()
    )


    duplicate_seqn = int(
        df["SEQN"]
        .duplicated()
        .sum()
    )


    print(
        f"{name}: "
        f"rows={len(df):,}, "
        f"unique SEQN={df['SEQN'].nunique():,}, "
        f"blank SEQN={blank_seqn:,}, "
        f"duplicate SEQN={duplicate_seqn:,}"
    )


    if blank_seqn != 0:

        raise ValueError(
            f"{name} contains blank SEQN."
        )


    if duplicate_seqn != 0:

        raise ValueError(
            f"{name} contains duplicate SEQN."
        )


print(
    "PASS: SEQN is unique in both Dietary files."
)


# =================================================================================================
# 8. VERIFY PARTICIPANT SETS
# =================================================================================================

#
# Total nutrient files should represent the same NHANES
# participant universe even though Day 2 dietary measurements
# may be missing for participants without a completed recall.
#

day1_seqn = set(
    day1["SEQN"]
)

day2_seqn = set(
    day2["SEQN"]
)


only_day1 = sorted(
    day1_seqn
    -
    day2_seqn
)

only_day2 = sorted(
    day2_seqn
    -
    day1_seqn
)


print(
    "\n--- SEQN SET COMPARISON ---"
)

print(
    "Only in Day 1:",
    len(only_day1)
)

print(
    "Only in Day 2:",
    len(only_day2)
)


if only_day1 or only_day2:

    raise ValueError(
        "DR1TOT_C and DR2TOT_C do not contain "
        "the same participant SEQN universe.\n\n"
        f"Only Day 1: {len(only_day1):,}\n"
        f"Only Day 2: {len(only_day2):,}"
    )


print(
    "PASS: Day 1 and Day 2 contain the same SEQN set."
)


# =================================================================================================
# 9. IDENTIFY OVERLAPPING COLUMN NAMES
# =================================================================================================

shared_columns_detected = [
    col
    for col in day1.columns
    if (
        col in day2.columns
        and
        col != "SEQN"
    )
]


print(
    "\n--- SHARED COLUMN NAMES ---"
)

print(
    shared_columns_detected
)


# =================================================================================================
# 10. EXPECTED SHARED NHANES VARIABLES
# =================================================================================================

#
# These variables intentionally appear in BOTH
# DR1TOT_C and DR2TOT_C.
#
# They describe shared Dietary/sample information rather
# than separate Day-1 vs Day-2 nutrient totals.
#

SHARED_COLUMNS = [
    "WTDRD1",
    "WTDR2D",
    "DRABF",
    "DRDINT"
]


if set(
    shared_columns_detected
) != set(
    SHARED_COLUMNS
):

    raise ValueError(
        "Unexpected shared-column pattern.\n\n"
        f"Expected:\n{SHARED_COLUMNS}\n\n"
        f"Detected:\n{shared_columns_detected}"
    )


print(
    "PASS: Exactly four expected shared variables found."
)


# =================================================================================================
# 11. ALIGN SHARED VARIABLES BY SEQN
# =================================================================================================

#
# We validate these BEFORE removing the duplicate Day 2 copies.
#

shared_comparison = (
    day1[
        ["SEQN"] + SHARED_COLUMNS
    ]
    .merge(
        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],
        on="SEQN",
        how="outer",
        suffixes=("_DAY1", "_DAY2"),
        validate="one_to_one"
    )
)


# =================================================================================================
# 12. VERIFY SHARED VARIABLES ARE EXACTLY IDENTICAL
# =================================================================================================

print(
    "\n--- SHARED VARIABLE FIDELITY CHECK ---"
)


for col in SHARED_COLUMNS:

    day1_col = (
        shared_comparison[
            f"{col}_DAY1"
        ]
    )


    day2_col = (
        shared_comparison[
            f"{col}_DAY2"
        ]
    )


    mismatch_mask = (
        day1_col
        !=
        day2_col
    )


    mismatch_count = int(
        mismatch_mask.sum()
    )


    print(
        f"{col}: "
        f"{mismatch_count:,} mismatches"
    )


    if mismatch_count != 0:

        examples = (
            shared_comparison.loc[
                mismatch_mask,
                [
                    "SEQN",
                    f"{col}_DAY1",
                    f"{col}_DAY2"
                ]
            ]
            .head(10)
        )


        print(
            examples
        )


        raise ValueError(
            f"{col} differs between "
            "DR1TOT_C and DR2TOT_C."
        )


print(
    "PASS: WTDRD1, WTDR2D, DRABF and DRDINT "
    "are exact matches between Day 1 and Day 2."
)


# =================================================================================================
# 13. REMOVE ONLY REDUNDANT DAY 2 COPIES
# =================================================================================================

#
# Keep canonical shared variables from Day 1.
#
# Remove the duplicate versions from Day 2 BEFORE merging.
#
# This prevents pandas from creating:
#
#     WTDRD1_DAY_2_TOTAL_INTAKE
#     WTDR2D_DAY_2_TOTAL_INTAKE
#     DRABF_DAY_2_TOTAL_INTAKE
#     DRDINT_DAY_2_TOTAL_INTAKE
#

day2_for_merge = (
    day2.drop(
        columns=SHARED_COLUMNS
    )
)


print(
    "\nRemoved redundant Day 2 copies:"
)

for col in SHARED_COLUMNS:

    print(
        " -",
        col
    )


# =================================================================================================
# 14. MERGE DAY 1 + DAY 2
# =================================================================================================

#
# ONE SEQN
#
# Day 1 information
#
# Day 2 information
#
# validate="one_to_one" guarantees the merge cannot silently
# multiply participants.
#

dietary = day1.merge(
    day2_for_merge,
    on="SEQN",
    how="outer",
    validate="one_to_one"
)


print(
    "\nPASS: Day 1 + Day 2 merge completed."
)


# =================================================================================================
# 15. VERIFY MERGED SEQN
# =================================================================================================

merged_duplicate_seqn = int(
    dietary["SEQN"]
    .duplicated()
    .sum()
)


merged_blank_seqn = int(
    dietary["SEQN"]
    .eq("")
    .sum()
)


if merged_duplicate_seqn != 0:

    raise ValueError(
        "Merged Dietary file contains duplicate SEQN."
    )


if merged_blank_seqn != 0:

    raise ValueError(
        "Merged Dietary file contains blank SEQN."
    )


if (
    dietary["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Merged Dietary SEQN contains '.0'."
    )


print(
    "PASS: Merged SEQN is unique and clean."
)


# =================================================================================================
# 16. VERIFY NO DUPLICATE COLUMN NAMES
# =================================================================================================

duplicate_column_names = (
    dietary.columns[
        dietary.columns.duplicated()
    ]
    .tolist()
)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found after merge:\n"
        + str(duplicate_column_names)
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 17. VERIFY ENGINEERED DUPLICATE COLUMNS DO NOT EXIST
# =================================================================================================

engineered_duplicate_patterns = [
    "_DAY_2_TOTAL_INTAKE",
    "_DAY2",
    "_X",
    "_Y"
]


bad_columns = []


for col in dietary.columns:

    upper_col = col.upper()


    if any(
        upper_col.endswith(pattern)
        for pattern in engineered_duplicate_patterns
    ):

        bad_columns.append(
            col
        )


if bad_columns:

    raise ValueError(
        "Unexpected engineered merge columns remain:\n"
        + str(bad_columns)
    )


print(
    "PASS: No engineered duplicate Dietary columns."
)


# =================================================================================================
# 18. CREATE LOGICAL COLUMN ORDER
# =================================================================================================

#
# User-requested structure:
#
#     SEQN
#
#     shared Dietary variables
#
#     Day 1 variables
#
#     Day 2 variables
#

shared_order = [
    "SEQN",
    "WTDRD1",
    "WTDR2D",
    "DRABF",
    "DRDINT"
]


#
# Preserve original Day 1 order.
#
day1_unique_columns = [
    col
    for col in day1.columns
    if col not in shared_order
]


#
# Preserve original Day 2 order.
#
day2_unique_columns = [
    col
    for col in day2.columns
    if (
        col != "SEQN"
        and
        col not in SHARED_COLUMNS
    )
]


final_column_order = (
    shared_order
    +
    day1_unique_columns
    +
    day2_unique_columns
)


# =================================================================================================
# 19. VERIFY COLUMN ORDER COVERS EVERYTHING
# =================================================================================================

if len(
    final_column_order
) != len(
    set(final_column_order)
):

    raise ValueError(
        "Column-order list itself contains duplicates."
    )


missing_from_order = set(
    dietary.columns
) - set(
    final_column_order
)


unexpected_in_order = set(
    final_column_order
) - set(
    dietary.columns
)


if missing_from_order:

    raise ValueError(
        "Columns missing from final ordering:\n"
        + str(
            sorted(
                missing_from_order
            )
        )
    )


if unexpected_in_order:

    raise ValueError(
        "Unknown columns in final ordering:\n"
        + str(
            sorted(
                unexpected_in_order
            )
        )
    )


dietary = dietary[
    final_column_order
]


print(
    "PASS: Final column order created."
)


# =================================================================================================
# 20. DISPLAY FINAL COLUMN STRUCTURE
# =================================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL DIETARY COLUMN STRUCTURE"
)

print(
    "=" * 100
)


print(
    "\nSHARED:"
)

for col in shared_order:

    print(
        " ",
        col
    )


print(
    "\nDAY 1 VARIABLES:",
    len(day1_unique_columns)
)


for col in day1_unique_columns:

    print(
        " ",
        col
    )


print(
    "\nDAY 2 VARIABLES:",
    len(day2_unique_columns)
)


for col in day2_unique_columns:

    print(
        " ",
        col
    )


# =================================================================================================
# 21. FINAL FIDELITY CHECK OF ORIGINAL DAY 1 COLUMNS
# =================================================================================================

#
# Every Day 1 value should survive the merge exactly.
#

print(
    "\n--- DAY 1 POST-MERGE FIDELITY ---"
)


day1_check = (
    day1.merge(
        dietary[
            day1.columns.tolist()
        ],
        on="SEQN",
        how="inner",
        suffixes=("_SOURCE", "_MERGED"),
        validate="one_to_one"
    )
)


day1_fidelity_failures = []


for col in day1.columns:

    if col == "SEQN":

        continue


    left = (
        day1_check[
            f"{col}_SOURCE"
        ]
    )

    right = (
        day1_check[
            f"{col}_MERGED"
        ]
    )


    mismatch_count = int(
        (
            left
            !=
            right
        )
        .sum()
    )


    if mismatch_count > 0:

        day1_fidelity_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day1_fidelity_failures:

    raise ValueError(
        "Day 1 values changed during merge:\n"
        + str(
            day1_fidelity_failures
        )
    )


print(
    "PASS: Every Day 1 value preserved exactly."
)


# =================================================================================================
# 22. FINAL FIDELITY CHECK OF DAY 2 UNIQUE COLUMNS
# =================================================================================================

print(
    "\n--- DAY 2 POST-MERGE FIDELITY ---"
)


day2_check_columns = (
    ["SEQN"]
    +
    day2_unique_columns
)


day2_check = (
    day2[
        day2_check_columns
    ]
    .merge(
        dietary[
            day2_check_columns
        ],
        on="SEQN",
        how="inner",
        suffixes=("_SOURCE", "_MERGED"),
        validate="one_to_one"
    )
)


day2_fidelity_failures = []


for col in day2_unique_columns:

    left = (
        day2_check[
            f"{col}_SOURCE"
        ]
    )


    right = (
        day2_check[
            f"{col}_MERGED"
        ]
    )


    mismatch_count = int(
        (
            left
            !=
            right
        )
        .sum()
    )


    if mismatch_count > 0:

        day2_fidelity_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day2_fidelity_failures:

    raise ValueError(
        "Day 2 values changed during merge:\n"
        + str(
            day2_fidelity_failures
        )
    )


print(
    "PASS: Every unique Day 2 value "
    "preserved exactly."
)


# =================================================================================================
# 23. DIETARY DECIMAL PROTECTION CHECK
# =================================================================================================

#
# Because the files were read with dtype=str,
# the Dietary decimal tokens should survive exactly.
#
# We do NOT convert to float.
#
# We do NOT round.
#

print(
    "\n--- DIETARY DECIMAL PROTECTION ---"
)


representative_decimal_columns = [
    "DR1TPROT",
    "DR1TP204",
    "DR2TPROT",
    "DR2TP204"
]


for col in representative_decimal_columns:

    if col not in dietary.columns:

        continue


    nonblank = int(
        dietary[col]
        .ne("")
        .sum()
    )


    decimal_tokens = int(
        dietary[col]
        .str.contains(
            ".",
            regex=False
        )
        .sum()
    )


    print(
        f"{col}: "
        f"nonblank={nonblank:,}, "
        f"tokens containing decimal point="
        f"{decimal_tokens:,}"
    )


print(
    "PASS: Dietary values were merged as text "
    "without rounding or numeric reformatting."
)


# =================================================================================================
# 24. FINAL SHAPE
# =================================================================================================

print(
    "\n--- FINAL MERGE SUMMARY ---"
)

print(
    "Rows:",
    f"{len(dietary):,}"
)

print(
    "Columns:",
    f"{len(dietary.columns):,}"
)

print(
    "Unique SEQN:",
    f"{dietary['SEQN'].nunique():,}"
)

print(
    "Duplicate SEQN:",
    int(
        dietary["SEQN"]
        .duplicated()
        .sum()
    )
)


# =================================================================================================
# 25. SAVE FINAL CSV
# =================================================================================================

#
# Since dataframe values are strings, pandas writes the
# corrected Dietary representation back without numeric
# conversion or float formatting.
#

dietary.to_csv(
    OUTPUT_PATH,
    index=False
)


print(
    "\nSaved to:"
)

print(
    OUTPUT_PATH
)


# =================================================================================================
# 26. RE-READ FINAL FILE AS TEXT
# =================================================================================================

final_check = pd.read_csv(
    OUTPUT_PATH,
    dtype=str,
    keep_default_na=False
)


# =================================================================================================
# 27. FINAL FILE VALIDATION
# =================================================================================================

if final_check.shape != dietary.shape:

    raise ValueError(
        "Saved CSV shape differs from merged dataframe."
    )


if (
    final_check.columns.tolist()
    !=
    dietary.columns.tolist()
):

    raise ValueError(
        "Saved CSV column order changed."
    )


if (
    final_check["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Saved CSV contains duplicate SEQN."
    )


if (
    final_check["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Saved CSV SEQN contains '.0'."
    )


# =================================================================================================
# 28. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

#
# Since we deliberately used text-mode processing,
# the saved CSV should reconstruct every field exactly.
#

text_fidelity_failures = []


for col in dietary.columns:

    same = (
        dietary[col]
        .eq(
            final_check[col]
        )
        .all()
    )


    if not same:

        text_fidelity_failures.append(
            col
        )


if text_fidelity_failures:

    raise ValueError(
        "Text fidelity changed during CSV export:\n"
        + str(
            text_fidelity_failures
        )
    )


print(
    "PASS: Saved CSV preserves all merged "
    "values exactly."
)


# =================================================================================================
# 29. FINAL RESULT
# =================================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "NHANES 2003-2004 DIETARY MERGE COMPLETE"
)

print(
    "=" * 100
)

print(
    "One SEQN per participant"
)

print(
    "Shared Dietary variables retained once:"
)

print(
    "  WTDRD1"
)

print(
    "  WTDR2D"
)

print(
    "  DRABF"
)

print(
    "  DRDINT"
)

print(
    "Day 1 variables appear before Day 2 variables"
)

print(
    "No engineered Day 2 duplicate copies"
)

print(
    "No duplicate SEQN"
)

print(
    "No SEQN '.0'"
)

print(
    "Dietary decimal text preserved AS IS"
)

print(
    "Day 1 fidelity: PASS"
)

print(
    "Day 2 fidelity: PASS"
)

print(
    "=" * 100
)

NHANES 2003-2004 DIETARY DAY 1 + DAY 2 MERGE
Day 1 shape: (9643, 160)
Day 2 shape: (9643, 81)
PASS: SEQN exists in both files.
PASS: SEQN contains no '.0' in either file.
DR1TOT_C: rows=9,643, unique SEQN=9,643, blank SEQN=0, duplicate SEQN=0
DR2TOT_C: rows=9,643, unique SEQN=9,643, blank SEQN=0, duplicate SEQN=0
PASS: SEQN is unique in both Dietary files.

--- SEQN SET COMPARISON ---
Only in Day 1: 0
Only in Day 2: 0
PASS: Day 1 and Day 2 contain the same SEQN set.

--- SHARED COLUMN NAMES ---
['WTDRD1', 'WTDR2D', 'DRABF', 'DRDINT']
PASS: Exactly four expected shared variables found.

--- SHARED VARIABLE FIDELITY CHECK ---
WTDRD1: 0 mismatches
WTDR2D: 0 mismatches
DRABF: 0 mismatches
DRDINT: 0 mismatches
PASS: WTDRD1, WTDR2D, DRABF and DRDINT are exact matches between Day 1 and Day 2.

Removed redundant Day 2 copies:
 - WTDRD1
 - WTDR2D
 - DRABF
 - DRDINT

PASS: Day 1 + Day 2 merge completed.
PASS: Merged SEQN is unique and clean.
PASS: No duplicate column names.
PASS: No engineered d